![Week 3 Day 2](../assets/banners/w3d2.png)

# Week 3 Day 2 - `create_agent`: Tools, Memory, Structured Responses and Middleware

Yesterday you wrote the ReAct loop by hand: call the model, run the tools it asks for, append the results, repeat. It was about fifteen lines, and it worked. But a real agent needs more: memory between turns, a structured answer your code can trust, a cap on how many calls it can make, logging, a way to keep student data private, and a human who can say "no".

Today you get all of that from **one function**: LangChain 1.x's `create_agent`. And then we'll lift the lid and discover that the agent it builds is... a LangGraph graph. That's your bridge to the rest of the week.

![Your 6-week roadmap](../assets/roadmap/week3.png)

### 🎯 By the end of this lab you will
1. 🤖 Build a CS201 rubric assistant with `create_agent` (model, tools, system prompt)
2. 🕸️ See that an agent **is a graph**, with `agent.get_graph().draw_mermaid()`
3. 🧠 Give it **memory** with a checkpointer and a `thread_id`
4. 📦 Get a validated Pydantic object back with `response_format` → `result["structured_response"]`
5. 🧅 Wrap it in **middleware**: your own logger, plus the built-ins for summarisation, retries, limits, PII and human approval

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 2 - `create_agent`: Tools, Memory and Structured Responses | 🧪 |
| 2 | Day 2 - Middleware: Logging, Summarization, Retries, Limits, PII, Human Approval | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Every idea today is shown twice: once with a <b>🔑 real model</b> (needs your API key, costs a fraction of a cent), and once with a <b>🎭 stand-in model</b> that replays a script, so the cell runs offline and always gives the same output. The stand-in never reads your prompt: it's a prop to show you the <i>mechanics</i>. The real model is where the intelligence is.</span>
        </td>
    </tr>
</table>

## Imports and environment

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
import os
import time

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ModelRetryMiddleware,
    PIIMiddleware,
    SummarizationMiddleware,
    ToolCallLimitMiddleware,
    before_model,
    wrap_tool_call,
)
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from pydantic import BaseModel, Field

from gradewise import datasets
from gradewise.config import DATA_DIR, settings
from gradewise.models import chat_model
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK: only the cells marked 🔑 need a key, the rest run offline.")
        print("   To set one up, see Part 5 of the setup guide (and remember to SAVE .env!)")

print(settings.describe_models())
print(datasets.describe())

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in exam.questions}
print(f"Loaded {exam.title}: {len(QUESTIONS)} questions")

## Part 1: An agent in one call

Two tools over Emma's real exam (the `get_rubric` from yesterday, plus a way to list the questions), and a system prompt. That's all `create_agent` needs.

In [ ]:
@tool
def list_questions() -> str:
    """List every question on the CS201 midterm with its ID, type and marks. Use this first if unsure of an ID."""
    return "\n".join(f"{q.id}: {q.question_type}, {q.max_marks:g} marks" for q in exam.questions)


@tool(parse_docstring=True)
def get_rubric(question_id: str) -> str:
    """Look up the marking rubric for one CS201 midterm question.

    Args:
        question_id: The question ID, e.g. "Q3" or "Q2a".
    """
    q = QUESTIONS.get(question_id)
    if q is None:
        return f"No question {question_id!r}. Valid IDs: {', '.join(QUESTIONS)}"
    return "\n".join(
        [f"{q.id} ({q.max_marks:g} marks): {q.text}"] + [f"- {c.id} [{c.marks:g}]: {c.description}" for c in q.criteria]
    )


TOOLS = [list_questions, get_rubric]
SYSTEM_PROMPT = (
    "You are GradeWise, Emma's CS201 marking assistant. "
    "Always look rubrics up with your tools; never guess marks. Be brief."
)

### 🔑 With a real model
`create_agent(model, tools, system_prompt=...)` returns a ready-to-run agent. You call it with a dict of `messages`, and get a dict back whose `messages` hold the whole conversation, tool calls included.

In [ ]:
agent = create_agent(chat_model(temperature=0), tools=TOOLS, system_prompt=SYSTEM_PROMPT)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Which question is worth the most marks, and what earns them?"}]}
)
for message in result["messages"]:
    message.pretty_print()

### 🎭 The same agent with a stand-in model
Here's the identical agent, driven by a scripted stand-in so you can see the full loop offline. Watch the message types: Human → AI (tool call) → Tool → AI (tool call) → Tool → AI (answer).

In [ ]:
# 🎭 A STAND-IN model, so you can watch agents work with no API key and no cost.
# 💡 GenericFakeChatModel replies with the messages we script, in order. It can't really call
#    tools, so bind_tools() just returns the same model: the tool calls are in our script.
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedModel(GenericFakeChatModel):
    """Replays a script of AIMessages. ⚠️ A teaching prop, not an LLM: it never reads your prompt."""

    def bind_tools(self, tools, **kwargs):
        return self


def scripted(*replies: AIMessage) -> ScriptedModel:
    return ScriptedModel(messages=iter(replies))


def call(name: str, call_id: str = "call_1", **args) -> AIMessage:
    """A scripted AIMessage that asks for ONE tool call, exactly like a real model would."""
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

In [ ]:
standin = scripted(
    call("list_questions"),
    call("get_rubric", call_id="call_2", question_id="Q5"),
    AIMessage(
        content="Q5 (paging, 5 marks) is joint highest with Q8: 1 mark each for offset bits, "
        "page-number bits, number of pages, the formula and the final size."
    ),
)
offline_agent = create_agent(standin, tools=TOOLS, system_prompt=SYSTEM_PROMPT)

result = offline_agent.invoke(
    {"messages": [{"role": "user", "content": "Which question is worth the most marks, and what earns them?"}]}
)
for message in result["messages"]:
    message.pretty_print()

## Part 2: An agent is a graph 🕸️

Here's the reveal. `create_agent` doesn't return some special "agent object": it returns a **compiled LangGraph graph**, the very same kind you'll build by hand tomorrow. Ask it to draw itself (this works offline):

In [ ]:
from gradewise.display import show_graph

print(type(offline_agent).__name__)
show_graph(offline_agent)

Paste that output into [mermaid.live](https://mermaid.live) to see it drawn. It's yesterday's loop, as a graph:

![Diagram](../assets/diagrams/c12da5461b95.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([__start__]) --> M[model]
  M -. tool calls .-> T[tools]
  T -.-> M
  M -. no tool calls .-> E([__end__])
```

</details>


💡 Two **nodes** (`model`, `tools`), and **conditional edges** (the dotted lines) that check "did the model ask for a tool?". Hold on to this picture: on day 3 you'll build graphs like it from scratch.

## Part 3: Memory with a checkpointer and a `thread_id`

Models are stateless: each call knows only the messages you send. To make an agent remember a conversation, give it a **checkpointer**. After every step it saves the graph's state, filed under a **`thread_id`** you choose. Same thread = same memory; new thread = a fresh start.

`InMemorySaver` keeps that in RAM (perfect for notebooks). On day 4 you'll swap in `SqliteSaver` so memory survives a restart.

In [ ]:
memory_agent = create_agent(
    scripted(
        AIMessage(content="Hi Emma! Noted: you're marking the CS201 midterm."),
        AIMessage(content="You're marking the CS201 midterm."),
        AIMessage(content="I don't know yet. What are you working on?"),
    ),
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),  # 💡 this one argument gives the agent memory
)

emma = {"configurable": {"thread_id": "emma-monday"}}
sam = {"configurable": {"thread_id": "sam-monday"}}

memory_agent.invoke({"messages": [{"role": "user", "content": "I'm Emma, marking the CS201 midterm."}]}, emma)
r1 = memory_agent.invoke({"messages": [{"role": "user", "content": "What am I marking?"}]}, emma)
r2 = memory_agent.invoke({"messages": [{"role": "user", "content": "What am I marking?"}]}, sam)

print(f"Emma's thread holds {len(r1['messages'])} messages → {r1['messages'][-1].text}")
print(f"Sam's thread holds  {len(r2['messages'])} messages → {r2['messages'][-1].text}")

💡 Notice we only sent the **new** message each time: the checkpointer loaded the rest. With the stand-in, the replies are scripted, but the message counts are real: Emma's thread has 4 messages, Sam's has 2.

🔑 Try it for real:

In [ ]:
real_memory = create_agent(
    chat_model(temperature=0), tools=TOOLS, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "emma-real"}}
real_memory.invoke(
    {"messages": [{"role": "user", "content": "Remember: I only care about the networking questions (Q6-Q8)."}]}, config
)
print(
    real_memory.invoke({"messages": [{"role": "user", "content": "Which of my questions is worth most?"}]}, config)[
        "messages"
    ][-1].text
)

## Part 4: Structured responses with `response_format`

In week 1 you used `with_structured_output` to get Pydantic objects from a single call. Agents can do it too: pass `response_format=YourModel`, and after the loop finishes, the validated object is waiting for you in `result["structured_response"]`.

In [ ]:
class QuestionBrief(BaseModel):
    """A short, structured brief on one exam question, for Emma's marking team."""

    question_id: str
    max_marks: float = Field(gt=0)
    criteria_count: int
    key_terms: list[str] = Field(description="3-6 terms a full-marks answer will mention")

🔑 With a real model, passing the class is enough. LangChain picks the best strategy for your provider: native structured output if the model supports it (`ProviderStrategy`), otherwise a hidden "respond" tool (`ToolStrategy`).

In [ ]:
brief_agent = create_agent(
    chat_model(temperature=0), tools=TOOLS, system_prompt=SYSTEM_PROMPT, response_format=QuestionBrief
)
result = brief_agent.invoke({"messages": [{"role": "user", "content": "Give me a brief on Q4b."}]})
result["structured_response"]

🎭 Offline, we ask for `ToolStrategy` explicitly: the "final answer" is then just one more tool call (named after the schema), which our stand-in can script.

In [ ]:
standin = scripted(
    call("get_rubric", question_id="Q4b"),
    call(
        "QuestionBrief",
        call_id="call_2",
        question_id="Q4b",
        max_marks=4,
        criteria_count=4,
        key_terms=["integer", "atomic", "wait", "signal", "binary", "counting"],
    ),
)
brief_agent = create_agent(
    standin, tools=TOOLS, system_prompt=SYSTEM_PROMPT, response_format=ToolStrategy(QuestionBrief)
)
result = brief_agent.invoke({"messages": [{"role": "user", "content": "Give me a brief on Q4b."}]})

brief = result["structured_response"]
print(type(brief).__name__, "→", brief)
assert brief.max_marks == QUESTIONS["Q4b"].max_marks  # 💡 it's a real Pydantic object: your code can check it

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The stand-in "model" claimed Q4b has 4 criteria and is worth 4 marks, and our <code>assert</code> checked the marks against the exam. What would happen if a real model returned <code>max_marks: -1</code>? (Hint: look at <code>Field(gt=0)</code>. The agent would get the validation error back and be asked to try again.)</span>
        </td>
    </tr>
</table>

## Part 5: Middleware

Middleware lets you **hook into the agent's loop** without rewriting it. Think of an onion 🧅: each middleware wraps the model call or the tool call, and can look at (or change) what goes in and what comes out.

![Diagram](../assets/diagrams/cafa9bc764ad.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([start]) --> BA[before_agent] --> BM[before_model] --> WM[[wrap_model_call → model]] --> AM[after_model]
  AM -- tool calls --> WT[[wrap_tool_call → tools]] --> BM
  AM -- done --> AA[after_agent] --> E([end])
```

</details>


| Hook | Runs | Good for |
|---|---|---|
| `@before_model` / `@after_model` | before / after every model call | trimming messages, checks, limits |
| `@wrap_model_call` | *around* the model call | retries, fallbacks, swapping the model |
| `@wrap_tool_call` | *around* each tool call | logging, timing, error handling |
| `@before_agent` / `@after_agent` | once per invocation | setup, final clean-up |

### Your own middleware: a tool-call logger
`@wrap_tool_call` gives you the `request` (with the tool call in `request.tool_call`) and a `handler` that runs the tool. You decide what happens around it:

In [ ]:
@wrap_tool_call
def log_tool_calls(request, handler):
    """Log every tool call, its arguments, its result and how long it took."""
    name, args = request.tool_call["name"], request.tool_call["args"]
    print(f"🛠️  → {name}({args})")
    start = time.perf_counter()
    result = handler(request)  # 💡 this actually runs the tool
    print(f"🛠️  ← {name} took {(time.perf_counter() - start) * 1000:.1f} ms: {result.content[:60]!r}…")
    return result


@before_model
def count_messages(state, runtime):
    print(f"🧠 calling the model with {len(state['messages'])} message(s)")
    return None  # 💡 None = change nothing


logged = create_agent(
    scripted(call("get_rubric", question_id="Q9"), AIMessage(content="Q9 is the ACID question: 1 mark per property.")),
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    middleware=[count_messages, log_tool_calls],
)
print(logged.invoke({"messages": [{"role": "user", "content": "What's Q9 about?"}]})["messages"][-1].text)

### The built-in middleware
LangChain 1.x ships with a toolbox of ready-made middleware in `langchain.agents.middleware`. These are the ones GradeWise cares about (checked against the installed version):

| Middleware | What it does | Key arguments |
|---|---|---|
| `SummarizationMiddleware` | Summarises old messages when the history gets long | `model`, `trigger=("messages", 50)` or `("tokens", 4000)`, `keep=("messages", 20)` |
| `ModelRetryMiddleware` / `ToolRetryMiddleware` | Retries failed calls with exponential backoff | `max_retries`, `initial_delay`, `backoff_factor` |
| `ModelCallLimitMiddleware` | Caps model calls per run or per thread | `run_limit`, `thread_limit`, `exit_behavior="end"` or `"error"` |
| `ToolCallLimitMiddleware` | Caps calls to one tool (or all tools) | `tool_name`, `run_limit`, `thread_limit` |
| `PIIMiddleware` | Finds personal data and redacts, masks, hashes or blocks it | `pii_type`, `strategy`, `detector` (a regex), `apply_to_input/output/tool_results` |
| `HumanInTheLoopMiddleware` | Pauses before chosen tools until a human approves, edits or rejects | `interrupt_on={"tool_name": True}` |
| `ModelFallbackMiddleware` | Falls back to another model if the first fails | `first_model, *additional_models` |

Let's try each one. Everything below runs offline.

#### 📜 Summarisation: keeping long conversations affordable
When the history passes the `trigger`, old messages are replaced by a summary written by the model you pass in (a cheap one, in real life). Here, a stand-in "summariser":

In [ ]:
summariser = scripted(*[AIMessage(content="Emma asked about several CS201 questions.")] * 5)
chatty = create_agent(
    scripted(*[AIMessage(content=f"Answer {i}") for i in range(5)]),
    tools=[],
    middleware=[SummarizationMiddleware(summariser, trigger=("messages", 6), keep=("messages", 2))],
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "long-chat"}}
for turn in range(5):
    messages = chatty.invoke({"messages": [{"role": "user", "content": f"Question {turn}"}]}, config)["messages"]
    print(f"turn {turn}: {len(messages)} messages, first one: {messages[0].content[:45]!r}")

💡 The history never grows past a handful of messages: when it hits 6, everything except the last 2 is folded into one summary message.

#### 🔁 Retries: surviving a network blip

In [ ]:
class FlakyModel(ScriptedModel):
    """🎭 Fails twice with a timeout, then works: just like a bad Wi-Fi day."""

    failures_left: int = 2

    def _generate(self, *args, **kwargs):
        if self.failures_left > 0:
            self.failures_left -= 1
            print("   💥 TimeoutError (simulated)")
            raise TimeoutError("simulated network blip")
        return super()._generate(*args, **kwargs)


resilient = create_agent(
    FlakyModel(messages=iter([AIMessage(content="Made it on the third try ✅")])),
    tools=[],
    middleware=[ModelRetryMiddleware(max_retries=3, initial_delay=0.1, jitter=False)],
)
print(resilient.invoke({"messages": [{"role": "user", "content": "hi"}]})["messages"][-1].text)

#### 🚦 Limits: an agent that can't run up your bill
This stand-in model is "stuck": it keeps asking for rubrics forever. Without a limit, a real agent like that would keep spending money until it hit LangGraph's recursion limit.

In [ ]:
def stuck_model():
    return scripted(*[call("get_rubric", call_id=f"c{i}", question_id=f"Q{i}") for i in range(1, 9)])


capped = create_agent(
    stuck_model(), tools=TOOLS, middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")]
)
print("ModelCallLimit →", capped.invoke({"messages": [{"role": "user", "content": "go"}]})["messages"][-1].text)

tool_capped = create_agent(
    stuck_model(),
    tools=TOOLS,
    middleware=[ToolCallLimitMiddleware(tool_name="get_rubric", run_limit=2), ModelCallLimitMiddleware(run_limit=4)],
)
for m in tool_capped.invoke({"messages": [{"role": "user", "content": "go"}]})["messages"]:
    if m.type == "tool":
        print("ToolCallLimit, tool result →", m.content[:70])

#### 🔒 PII: keeping student data out of the model
Student names, emails and IDs are personal data. `PIIMiddleware` scans messages *before* they reach the model. It knows `email`, `credit_card`, `ip`, `mac_address` and `url` out of the box, and you can add your own type with a regex, like Northbridge's student IDs:

In [ ]:
private = create_agent(
    scripted(AIMessage(content="Sure, I'll look at that student's Q3.")),
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact"),
        PIIMiddleware("student_id", detector=r"NB-\d{4}-\d{4}", strategy="mask"),
    ],
)
result = private.invoke(
    {"messages": [{"role": "user", "content": "Sam (NB-2026-0142, sam@northbridge.edu) wants his Q3 re-marked."}]}
)
print("What the model saw:", result["messages"][0].content)

#### 🙋 Human in the loop: approve, edit or reject
Some tools are too important to run unsupervised. `HumanInTheLoopMiddleware` **pauses the agent** before chosen tools run, and the human can **approve**, **edit** the arguments, **reject** with a reason, or **respond** in the tool's place. The pause is a LangGraph `interrupt` (day 4's star), so it needs a checkpointer. You resume with a `Command` carrying your decision.

In [ ]:
@tool
def flag_for_review(question_id: str, reason: str) -> str:
    """Flag a student's answer for Emma to re-mark by hand. Use when unsure."""
    return f"🚩 Flagged {question_id} for Emma: {reason}"


def reviewing_agent():
    return create_agent(
        scripted(call("flag_for_review", question_id="Q3", reason="handwriting unclear"), AIMessage(content="Done.")),
        tools=[flag_for_review],
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"flag_for_review": True})],
        checkpointer=InMemorySaver(),  # ⚠️ pausing needs a checkpointer
    )


hitl = reviewing_agent()
config = {"configurable": {"thread_id": "review-1"}}
paused = hitl.invoke({"messages": [{"role": "user", "content": "Mark Sam's Q3."}]}, config)

request = paused["__interrupt__"][0].value
print(
    "⏸  Paused! The agent wants to run:", request["action_requests"][0]["name"], request["action_requests"][0]["args"]
)
print("   Emma may:", request["review_configs"][0]["allowed_decisions"])

In [ ]:
# ▶️ Emma approves, and the agent carries on from exactly where it paused
done = hitl.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print(done["messages"][-2].content)

# 🧪 TRY: or she edits the arguments, or rejects with a message the model will read
for decision in [
    {
        "type": "edit",
        "edited_action": {
            "name": "flag_for_review",
            "args": {"question_id": "Q3", "reason": "Emma: check circular wait"},
        },
    },
    {"type": "reject", "message": "No need: the answer is clearly legible."},
]:
    agent_again, cfg = reviewing_agent(), {"configurable": {"thread_id": decision["type"]}}
    agent_again.invoke({"messages": [{"role": "user", "content": "Mark Sam's Q3."}]}, cfg)
    final = agent_again.invoke(Command(resume={"decisions": [decision]}), cfg)
    print(f"{decision['type']:6} → …{final['messages'][-2].content[-75:]}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Order matters.</b> Middleware runs in the order you list it for the "before" hooks, and wraps from the outside in (the first in the list is the outermost layer). Put <code>PIIMiddleware</code> before anything that logs messages, or your logs will contain the very data you were hiding. And always add a <code>ModelCallLimitMiddleware</code> to an agent that can loop: it's a seatbelt for your bill.</span>
        </td>
    </tr>
</table>

### 🔑 Everything together, with a real model
A production-shaped agent: real model, memory, structured output, logging, a call limit and privacy.

In [ ]:
assistant = create_agent(
    chat_model(temperature=0),
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    response_format=QuestionBrief,
    checkpointer=InMemorySaver(),
    middleware=[
        PIIMiddleware("student_id", detector=r"NB-\d{4}-\d{4}", strategy="mask"),
        ModelCallLimitMiddleware(run_limit=6),
        log_tool_calls,
    ],
)
result = assistant.invoke(
    {
        "messages": [
            {"role": "user", "content": "Student NB-2026-0142 asked about the deadlock question. Brief me on it."}
        ]
    },
    {"configurable": {"thread_id": "emma-tuesday"}},
)
result["structured_response"]

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Middleware is where an agent becomes something a school can actually sign off on. PII redaction is a GDPR / FERPA conversation you can win; call limits turn "it might cost anything" into a fixed worst case; and human-in-the-loop means a person stays accountable for every mark. Each one is a single line, and that's a strong story to tell a client.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">In week 5, GradeWise's grader is built with <code>create_agent</code>: tools <code>retrieve_reference</code>, <code>get_rubric</code> and <code>flag_for_review</code>, <code>response_format=GradeResult</code> (from <code>gradewise/schemas/grading.py</code>), and middleware for limits and retries. Open <code>gradewise/schemas/grading.py</code> now and compare <code>GradeResult</code> with today's <code>QuestionBrief</code>: same idea, more fields.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🤖 built a CS201 rubric assistant with `create_agent(model, tools, system_prompt=...)`
- 🕸️ discovered that an agent **is** a compiled LangGraph graph (`model` ⇄ `tools`)
- 🧠 gave it memory with `checkpointer=InMemorySaver()` and a `thread_id`
- 📦 got validated Pydantic objects back via `response_format` → `result["structured_response"]`
- 🧅 wrapped it in middleware: your own `@wrap_tool_call` logger, plus summarisation, retries, limits, PII and human approval

**Tomorrow (Day 3):** we stop letting `create_agent` draw the graph and start drawing our own. State, nodes, edges, reducers and supersteps, then your first `StateGraph` with routing and a loop, all offline.

⏭ **Next:** `week3/day3.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Build a <b>"marks auditor"</b> agent. Give it a new tool <code>sum_criteria(question_id)</code> that returns a question's criteria total and max marks, plus <code>list_questions</code>. Ask for a structured answer <code>AuditReport(checked: list[str], all_add_up: bool, problems: list[str])</code>. Add a custom <code>@wrap_tool_call</code> middleware that <b>counts</b> tool calls per tool name (in a dict), and a <code>ToolCallLimitMiddleware</code> so <code>sum_criteria</code> can run at most 5 times. Run it with a stand-in script (and 🔑 a real model). Solution: <code>week3/solutions/day2_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangChain agents: <a href="https://docs.langchain.com/oss/python/langchain/agents">docs.langchain.com/oss/python/langchain/agents</a><br/> • Structured output: <a href="https://docs.langchain.com/oss/python/langchain/structured-output">docs.langchain.com/oss/python/langchain/structured-output</a><br/> • Middleware overview and built-ins: <a href="https://docs.langchain.com/oss/python/langchain/middleware">docs.langchain.com/oss/python/langchain/middleware</a><br/> • Short-term memory: <a href="https://docs.langchain.com/oss/python/langchain/short-term-memory">docs.langchain.com/oss/python/langchain/short-term-memory</a></span>
        </td>
    </tr>
</table>